<a href="https://colab.research.google.com/github/lalawee/special-octo-barnacle/blob/main/TrainingVerificationScript_for_exercise5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gym
import numpy as np
import torch
import torch.nn as nn

In [ ]:
#Step 1: the PolicyModel class before loading

class PolicyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(4, 20)
        self.linear2 = nn.Linear(20, 30)
        self.linear3 = nn.Linear(30, 2)
        self.activation = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.saved_log_probs = []
        self.rewards = []

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        x = self.activation(x)
        x = self.linear3(x)
        x = self.softmax(x)
        return x

In [ ]:
# Step 2: Mount Drive and load the trained model
from google.colab import drive
drive.mount('/content/drive')

model = torch.load('/content/drive/MyDrive/policyNet.pt')
model.eval()  # Set to evaluation mode (no training)

# Step 3: Set up environment
env = gym.make("CartPole-v1", new_step_api=True, render_mode="human")

# Step 4: Run 100 episodes using the trained model
Total_Rewards = 0

In [ ]:
for num_episode in range(1, 101):
    rewards = 0
    s = env.reset()
    terminated = False
    truncated = False

    while (not terminated) and (not truncated):
        # Convert numpy state to torch tensor
        state_tensor = torch.from_numpy(s).float().unsqueeze(0)

        # Get action probabilities from model
        with torch.no_grad():  # No gradient needed for inference
            probs = model(state_tensor)

        # Pick the action with highest probability (greedy)
        a = torch.argmax(probs, dim=1).item()

        s, r, terminated, truncated, _ = env.step(a)
        rewards += r

    Total_Rewards += rewards
    if num_episode % 10 == 0:
        print(f"Episode {num_episode}: {rewards}")

print("Average", Total_Rewards / 100)